Komórka 1: importy i połączenie

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent / "src"))  # żeby zaimportować ml/src/db.py
from db import get_engine

engine = get_engine()

Komórka 2: wczytanie notowań

In [2]:
prices = pd.read_sql(
    "SELECT ticker, date, close, volume FROM prices ORDER BY ticker, date",
    engine,
    parse_dates=["date"],
)
prices["close"] = prices["close"].astype(float)  # NUMERIC z Postgresa przychodzi jako Decimal
prices.info()
prices.head()

<class 'pandas.DataFrame'>
RangeIndex: 208290 entries, 0 to 208289
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype        
---  ------  --------------   -----        
 0   ticker  208290 non-null  str          
 1   date    208290 non-null  datetime64[s]
 2   close   208290 non-null  float64      
 3   volume  208290 non-null  int64        
dtypes: datetime64[s](1), float64(1), int64(1), str(1)
memory usage: 7.0 MB


,ticker,date,close,volume
0,ABE,2010-01-04,11.5005,2774
1,ABE,2010-01-05,11.6124,8251
2,ABE,2010-01-06,12.1723,30726
3,ABE,2010-01-07,12.1003,4145
4,ABE,2010-01-08,12.6201,29859


Komórka 3: z formatu długiego na szeroki

In [3]:
close = prices.pivot(index="date", columns="ticker", values="close")
turnover = prices.assign(t=prices["close"] * prices["volume"]).pivot(
    index="date", columns="ticker", values="t"
)
print(close.shape)
close.iloc[-5:, :8]

(4210, 60)


ticker,ABE,ACP,ALE,ALR,APR,ASB,ASE,ATT
date,,,,,,,,
2026-10-27,150.4,250.5,58.80,135.00,28.45,165.2,76.0,17.82
2026-10-28,153.4,246.8,59.07,137.50,28.10,163.5,76.7,17.35
2026-10-29,153.6,247.8,59.90,142.00,28.70,168.2,77.0,17.12
2026-10-30,148.0,244.5,59.55,133.35,28.20,160.0,75.1,16.66
2026-11-02,145.8,249.1,60.00,134.30,27.45,157.2,73.4,16.75


Komórka 4: ceny na koniec miesiąca

In [4]:
monthly_close = close.resample("ME").last()
print(monthly_close.shape)
monthly_close.tail(3).iloc[:, :8]

(203, 60)


ticker,ABE,ACP,ALE,ALR,APR,ASB,ASE,ATT
date,,,,,,,,
2026-09-30,151.4,219.0,50.63,137.70,29.95,184.9,75.8,19.07
2026-10-31,148.0,244.5,59.55,133.35,28.20,160.0,75.1,16.66
2026-11-30,145.8,249.1,60.00,134.30,27.45,157.2,73.4,16.75


Komórka 5: usunięcie niepełnego miesiąca

In [5]:
last_day = close.index.max()
if last_day < monthly_close.index[-1] - pd.Timedelta(days=4):
    print(f"Usuwam niepełny miesiąc {monthly_close.index[-1].date()} (dane do {last_day.date()})")
    monthly_close = monthly_close.iloc[:-1]
monthly_close.index[-3:]

Usuwam niepełny miesiąc 2026-11-30 (dane do 2026-11-02)


DatetimeIndex(['2026-08-31', '2026-09-30', '2026-10-31'], dtype='datetime64[s]',
              name='date', freq='ME')

Komórka 6: cechy

In [6]:
daily_ret = close / close.shift(1) - 1

def at_month_end(df):
    """Bierze wartość z ostatniego dnia każdego miesiąca, wyrównaną do monthly_close."""
    return df.resample("ME").last().reindex(monthly_close.index)

features = {
    # momentum: zmiana ceny w ostatnich 1, 3, 6 i 12 miesiącach
    "mom_1m": monthly_close / monthly_close.shift(1) - 1,
    "mom_3m": monthly_close / monthly_close.shift(3) - 1,
    "mom_6m": monthly_close / monthly_close.shift(6) - 1,
    "mom_12m": monthly_close / monthly_close.shift(12) - 1,
    # zmienność: odchylenie dziennych stóp zwrotu z ~3 miesięcy, w skali roku
    "vol_3m": at_month_end(daily_ret.rolling(63, min_periods=50).std() * np.sqrt(252)),
    # odległość od najwyższej ceny z ostatnich 12 miesięcy (0 = na szczycie, -0.3 = 30% poniżej)
    "dist_high_12m": at_month_end(close / close.rolling(252, min_periods=200).max() - 1),
    # płynność: logarytm średniego dziennego obrotu w zł z ~3 miesięcy
    "liquidity_3m": at_month_end(np.log(turnover.rolling(63, min_periods=50).mean())),
}

{name: f.notna().sum().sum() for name, f in features.items()}

{'mom_1m': np.int64(10060),
 'mom_3m': np.int64(9937),
 'mom_6m': np.int64(9753),
 'mom_12m': np.int64(9390),
 'vol_3m': np.int64(9805),
 'dist_high_12m': np.int64(9414),
 'liquidity_3m': np.int64(9871)}

Komórka 7: etykieta

In [11]:
MIN_TURNOVER = 100_000  # minimalny średni dzienny obrót w zł z ostatnich ~3 miesięcy
investable = features["liquidity_3m"] >= np.log(MIN_TURNOVER)

print("Odsetek par spółka-miesiąc spełniających filtr:", round(investable.stack().mean(), 3))

fwd_ret = (monthly_close.shift(-1) / monthly_close - 1).where(investable)  # tylko płynne spółki
median = fwd_ret.median(axis=1)                                          # mediana płynnego uniwersum
label = fwd_ret.gt(median, axis=0).astype(float).where(fwd_ret.notna())

print("Średnia etykieta:", round(label.stack().mean(), 3))

Odsetek par spółka-miesiąc spełniających filtr: 0.755
Średnia etykieta: 0.495


Komórka 8: zbiór w formacie długim

In [12]:
dataset = pd.concat({name: f.stack() for name, f in features.items()}, axis=1)
dataset["fwd_ret"] = fwd_ret.stack()
dataset["label"] = label.stack()
dataset.index.names = ["date", "ticker"]

FEATURES = list(features)
train = dataset.dropna(subset=FEATURES + ["label"])
live = dataset.loc[dataset.index.get_level_values("date") == monthly_close.index[-1]].dropna(subset=FEATURES)
live = live[live["liquidity_3m"] >= np.log(MIN_TURNOVER)]

print("Zbiór treningowy:", train.shape)
print("Miesięcy:", train.index.get_level_values("date").nunique(),
      "| spółek:", train.index.get_level_values("ticker").nunique())
print("Udział etykiety 1:", round(train["label"].mean(), 3))
print("Ostatni miesiąc do predykcji:", monthly_close.index[-1].date(), "| spółek:", len(live))
train[FEATURES].describe().T.round(3)

Zbiór treningowy: (8555, 9)
Miesięcy: 189 | spółek: 60
Udział etykiety 1: 0.494
Ostatni miesiąc do predykcji: 2026-10-31 | spółek: 60


,count,mean,std,min,25%,50%,75%,max
mom_1m,8555.0,0.016,0.108,-0.583,-0.046,0.008,0.068,0.914
mom_3m,8555.0,0.051,0.204,-0.727,-0.067,0.030,0.144,2.078
mom_6m,8555.0,0.107,0.328,-0.783,-0.084,0.064,0.245,4.999
mom_12m,8555.0,0.233,0.578,-0.860,-0.109,0.123,0.439,6.611
vol_3m,8555.0,0.361,0.154,0.022,0.268,0.328,0.407,2.176
dist_high_12m,8555.0,-0.165,0.153,-0.879,-0.241,-0.122,-0.049,0.000
liquidity_3m,8555.0,15.124,1.892,11.513,13.669,15.083,16.541,19.785


Komórka 9: zapis

In [13]:
data_dir = Path.cwd().parent / "data"
data_dir.mkdir(exist_ok=True)
dataset.to_parquet(data_dir / "dataset.parquet")
print("Zapisano:", data_dir / "dataset.parquet")

Zapisano: /Users/bartek/Desktop/portfolio_ml/ml/data/dataset.parquet


In [15]:
def show_extremes(col, n=3):
    print(f"\n{col} - najwyższe:")
    print(train[col].nlargest(n).round(3).to_string())
    print(f"{col} - najniższe:")
    print(train[col].nsmallest(n).round(3).to_string())

for col in ["vol_3m","mom_12m", "liquidity_3m"]:
    show_extremes(col)


vol_3m - najwyższe:
date        ticker
2022-04-30  LBW       2.176
2022-03-31  LBW       2.117
2020-05-31  LBW       1.853
vol_3m - najniższe:
date        ticker
2012-10-31  BNP       0.022
2011-01-31  EBP       0.069
2011-03-31  EBP       0.071

mom_12m - najwyższe:
date        ticker
2017-01-31  JSW       6.611
2011-04-30  CDR       6.575
2011-05-31  CDR       6.087
mom_12m - najniższe:
date        ticker
2020-03-31  MDV      -0.860
2022-07-31  TEN      -0.813
2013-09-30  PXM      -0.805

liquidity_3m - najwyższe:
date        ticker
2021-01-31  CDR       19.785
2026-03-31  KGH       19.775
2021-02-28  CDR       19.758
liquidity_3m - najniższe:
date        ticker
2012-06-30  LBW       11.513
2012-11-30  LBW       11.515
2019-08-31  CBF       11.515
